In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .config("spark.driver.extraJavaOptions", "-Djava.library.path=C:/tools/hadoop/bin") \
    .getOrCreate()

In [3]:
# spark.sparkContext.uiWebUrl

In [4]:
df_green = spark.read.parquet('../data/pq/green/*/*')

In [5]:
# df_green.columns

In [6]:
df_green.createOrReplaceTempView('green')

In [7]:
df_green_revenue = spark.sql("""
SELECT
    date_trunc('hour', lpep_pickup_datetime) AS hour,
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    count(1) as number_records
FROM
    green
where
    lpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
order by
    1, 2
""")

In [8]:
# df_green_revenue.show()

In [9]:
df_green_revenue \
    .repartition(20) \
    .write.parquet('../data/report/revenue/green', mode='overwrite')

In [10]:
df_yellow = spark.read.parquet('../data/pq/yellow/*/*')
df_yellow.createOrReplaceTempView('yellow')

In [11]:
df_yellow_revenue = spark.sql("""
SELECT
    date_trunc('hour', tpep_pickup_datetime) AS hour,
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    count(1) as number_records
FROM
    yellow
where
    tpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
order by
    1, 2
""")

In [12]:
df_green_revenue = spark.read.parquet('../data/report/revenue/green')
df_yellow_revenue = spark.read.parquet('../data/report/revenue/yellow')

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/D:/data-engineering/data-engineering-zoomcamp/batch/data/report/revenue/yellow. SQLSTATE: 42K03

In [ ]:
df_yellow_revenue \
    .repartition(20) \
    .write.parquet('../data/report/revenue/yellow', mode='overwrite')

In [ ]:
df_green_revenue_tmp = df_green_revenue \
    .withColumnRenamed('amount', 'green_amount') \
    .withColumnRenamed('number_records', 'green_number_records')

df_yellow_revenue_tmp = df_yellow_revenue \
    .withColumnRenamed('amount', 'yellow_amount') \
    .withColumnRenamed('number_records', 'yellow_number_records')

In [ ]:
df_join = df_green_revenue_tmp.join(df_yellow_revenue_tmp, on=['hour', 'zone'], how='outer')

In [ ]:
df_join.write.parquet('../data/report/revenue/total', mode='overwrite')

In [ ]:
# df_join.show()

In [ ]:
df_join = spark.read.parquet('../data/report/revenue/total')

In [ ]:
# !curl -fL --create-dirs -o ../zones/taxi_zone_lookup.csv https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

In [ ]:
df_zones = spark.read \
    .option("header", "true") \
    .csv('../zones/taxi_zone_lookup.csv')

In [ ]:
df_zones.write.parquet('../data/zones/', mode='overwrite')

In [ ]:
df_zones = spark.read.parquet('../data/zones/')

In [ ]:
# df_zones.show()

In [ ]:
df_result = df_join.join(df_zones, df_join.zone == df_zones.LocationID)

In [ ]:
df_result.drop('LocationID', 'zone').write.parquet('../tmp/revenue-zones')